# Spark SQL

---
## 1. Environment Setup

Spark SQL is the SQL interface to Apache Spark. Under the hood every SQL query is compiled into the same directed acyclic graph (DAG) of operations as a programmatic DataFrame query.

The cells below install – as usual – PySpark and Java (required as Spark runs on the JVM), then start a **`SparkSession`**, the single entry point to all Spark functionality. We use `master("local[*]")` to run locally using all available CPU cores.


In [ ]:
!pip install pyspark

# Install Java 17 (required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

# Set JAVA_HOME so PySpark can find the JVM
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkSQL_Class")
    # Tell Spark to push down filters into data sources whenever possible
    .config("spark.sql.pushdown.enabled", "true")
    .getOrCreate()
)

### 1.1 Schemas in Spark

A **schema** describes the structure of a DataFrame: the name and data type of every column, and whether each column may contain `null` values. Spark represents a schema as a `StructType` containing a list of `StructField` objects.

```python
StructType([
    StructField("price",    IntegerType(), nullable=True),
    StructField("area",     IntegerType(), nullable=True),
    StructField("bedrooms", IntegerType(), nullable=True),
])
```

**Why schemas matter**

* **Correctness** — without an explicit schema, Spark reads every column as a string and then tries to infer types. Inference can be wrong (e.g. a column of integers that occasionally contains `"N/A"` will be inferred as `StringType`).
* **Performance** — schema inference requires a full extra pass over the data. For large files this is expensive.
* **Stability** — in production pipelines, an explicit schema makes the code self-documenting and prevents surprises when upstream data changes slightly.

You can provide a schema in two ways:

| Approach | How | When to use |
|----------|-----|-------------|
| `inferSchema=True` | Spark scans the data and guesses types | Exploratory work on small files |
| Explicit `StructType` | You define every column up front | Production pipelines, large files |

In this notebook we use `inferSchema=True` for convenience, then inspect the result with `printSchema()`.


In [ ]:
housing_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("housing.csv")
)

# printSchema() shows the inferred column names, types, and nullability
housing_df.printSchema()


The schema above was inferred automatically. If we wanted to enforce it explicitly — for example in a production job — we would write:

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("price",             IntegerType(), True),
    StructField("area",              IntegerType(), True),
    StructField("bedrooms",          IntegerType(), True),
    StructField("bathrooms",         IntegerType(), True),
    StructField("stories",           IntegerType(), True),
    StructField("mainroad",          StringType(),  True),
    StructField("guestroom",         StringType(),  True),
    StructField("basement",          StringType(),  True),
    StructField("hotwaterheating",   StringType(),  True),
    StructField("airconditioning",   StringType(),  True),
    StructField("parking",           IntegerType(), True),
    StructField("prefarea",          StringType(),  True),
    StructField("furnishingstatus",  StringType(),  True),
])

housing_df = spark.read.schema(schema).option("header", "true").csv("housing.csv")
```

Once we have a DataFrame, we register it as a **temporary view** so SQL queries can reference it by name.


In [ ]:
housing_df.createOrReplaceTempView("housing")

---
## 2. Basic SQL Queries

`spark.sql(...)` accepts any ANSI SQL string and returns a **DataFrame**.
`.show()` triggers evaluation.


In [ ]:
# SELECT a few columns
spark.sql("""
    SELECT price, area, bedrooms, bathrooms, furnishingstatus
    FROM   housing
    LIMIT  10
""").show()


In [ ]:
# COUNT rows and basic statistics
spark.sql("""
    SELECT
        COUNT(*)              AS total_rows,
        ROUND(AVG(price), 0)  AS avg_price,
        MIN(price)            AS min_price,
        MAX(price)            AS max_price
    FROM housing
""").show()


---
## 3. Filtering, Aggregation, and Sorting

Standard SQL clauses — `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY` — work exactly as in any relational database.


In [ ]:
# Furnished houses on the main road, ordered by price
spark.sql("""
    SELECT price, area, bedrooms, bathrooms, parking
    FROM   housing
    WHERE  furnishingstatus = 'furnished'
      AND  mainroad = 'yes'
    ORDER  BY price DESC
    LIMIT  10
""").show()


In [ ]:
# Average price per furnishing status (only groups with more than 50 houses)
spark.sql("""
    SELECT
        furnishingstatus,
        COUNT(*)                    AS num_houses,
        ROUND(AVG(price), 0)        AS avg_price,
        ROUND(AVG(area),  0)        AS avg_area_sqft
    FROM   housing
    GROUP  BY furnishingstatus
    HAVING COUNT(*) > 50
    ORDER  BY avg_price DESC
""").show()


In [ ]:
# CASE expressions — price tier classification
spark.sql("""
    SELECT
        CASE
            WHEN price < 3500000  THEN 'Budget'
            WHEN price < 6000000  THEN 'Mid-range'
            WHEN price < 9000000  THEN 'Premium'
            ELSE                       'Luxury'
        END               AS price_tier,
        COUNT(*)          AS count,
        ROUND(AVG(area),0) AS avg_area_sqft
    FROM   housing
    GROUP  BY price_tier
    ORDER  BY avg_area_sqft DESC
""").show()


---
## 4. Joins

Spark SQL supports `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`, `CROSS`, and `SEMI / ANTI` joins.

We will create a small **amenities reference table** to illustrate a join.


In [ ]:
# Create a lookup table for amenity scores
amenities_data = [
    ("furnished",       3, "All appliances and furniture provided"),
    ("semi-furnished",  2, "Basic furniture only"),
    ("unfurnished",     1, "Empty shell — bring your own"),
]
amenities_schema = ["furnishingstatus", "amenity_score", "description"]

amenities_df = spark.createDataFrame(amenities_data, schema=amenities_schema)
amenities_df.createOrReplaceTempView("amenities")
amenities_df.show(truncate=False)


In [ ]:
spark.sql("""
    SELECT
        h.price,
        h.area,
        h.bedrooms,
        h.furnishingstatus,
        a.amenity_score,
        a.description
    FROM   housing   h
    JOIN   amenities a USING (furnishingstatus)
    ORDER  BY h.price DESC
    LIMIT  8
""").show(truncate=False)


In [ ]:
# LEFT JOIN — every house is kept, even if it had no match in the amenities table
spark.sql("""
    SELECT
        h.price,
        h.furnishingstatus,
        COALESCE(a.amenity_score, 0) AS amenity_score
    FROM   housing   h
    LEFT   JOIN amenities a USING (furnishingstatus)
    LIMIT  5
""").show()


---
## 5. Common Table Expressions (CTEs)

A **Common Table Expression** (CTE) is a named, temporary result set defined within a query using the `WITH` keyword. It exists only for the duration of the query that defines it and can be referenced by name in the main `SELECT` that follows.

```sql
WITH cte_name AS (
    SELECT ...
    FROM   ...
    WHERE  ...
)
SELECT *
FROM   cte_name
WHERE  ...
```

**Why use CTEs?**

* **Readability** — instead of deeply nested subqueries, each logical step gets its own named block.
* **Reusability** — the same CTE can be referenced multiple times in one query, avoiding repetition.
* **Debugging** — you can test each CTE individually by temporarily turning it into a standalone query.

Multiple CTEs can be chained by separating them with commas:

```sql
WITH
step1 AS (SELECT ...),
step2 AS (SELECT ... FROM step1)
SELECT * FROM step2
```

Spark treats CTEs as logical views — they do not materialise data to disk and are re-evaluated each time they are referenced unless you explicitly cache the result.


In [ ]:
# Example: use a CTE to compute average price per bedroom count,
# then filter for only the above-average tiers
spark.sql("""
    WITH avg_by_bedrooms AS (
        SELECT
            bedrooms,
            ROUND(AVG(price), 0)  AS avg_price,
            COUNT(*)              AS num_houses
        FROM   housing
        GROUP  BY bedrooms
    ),
    overall AS (
        SELECT ROUND(AVG(price), 0) AS overall_avg FROM housing
    )
    SELECT
        a.bedrooms,
        a.avg_price,
        a.num_houses,
        o.overall_avg,
        ROUND((a.avg_price - o.overall_avg) / o.overall_avg * 100, 1) AS pct_above_avg
    FROM   avg_by_bedrooms a
    CROSS  JOIN overall o
    WHERE  a.avg_price > o.overall_avg
    ORDER  BY a.avg_price DESC
""").show()


---
## 6. Caching Tables

By default Spark recomputes a DataFrame from scratch every time it is referenced (lazy evaluation).
**Caching** materialises the data in memory so repeated queries skip the recomputation cost.

| Method | Storage | When to use |
|--------|---------|-------------|
| `CACHE TABLE t` | Memory (serialized by default) | Iterative SQL queries on the same table |
| `CACHE TABLE t OPTIONS('storageLevel' 'DISK_ONLY')` | Disk | Dataset too large for RAM |
| `UNCACHE TABLE t` | — | Free memory when done |

> ⚠️ Caching is only beneficial when a table is queried **multiple times** in the same session.
> A single-pass ETL job should not cache — it wastes RAM and serialisation time.


In [ ]:
import time

# Without cache
t0 = time.time()
for _ in range(3):
    spark.sql("SELECT COUNT(*) FROM housing WHERE price > 5000000").collect()
no_cache_time = time.time() - t0
print(f"Without cache: {no_cache_time:.3f}s for 3 runs")

# With cache
spark.sql("CACHE TABLE housing")

t0 = time.time()
for _ in range(3):
    spark.sql("SELECT COUNT(*) FROM housing WHERE price > 5000000").collect()
cache_time = time.time() - t0
print(f"With cache:    {cache_time:.3f}s for 3 runs")

spark.sql("UNCACHE TABLE housing")


In [ ]:
# Inspect what is currently cached
spark.sql("SHOW TABLES").show()


---
## 7. Partitioning

**Partitioning** splits data into separate directories on disk, one per distinct value of the partition column.
Queries filtering on that column skip irrelevant directories entirely — this is called *partition pruning* and can reduce I/O by orders of magnitude on large datasets.

**Bucketing** pre-sorts data into a fixed number of files by a hash of a column, which eliminates the shuffle needed for joins and aggregations on that column.

```
partitioned table layout on disk
housing_partitioned/
    furnishingstatus=furnished/
        part-0000.parquet
    furnishingstatus=semi-furnished/
        part-0000.parquet
    furnishingstatus=unfurnished/
        part-0000.parquet
```


In [ ]:
import shutil, os

OUT = "/tmp/housing_partitioned"
shutil.rmtree(OUT, ignore_errors=True)

# Write as Parquet partitioned by furnishingstatus
(
    housing_df
    .write
    .partitionBy("furnishingstatus")
    .mode("overwrite")
    .parquet(OUT)
)

# Read back — Spark discovers the partitions automatically
partitioned_df = spark.read.parquet(OUT)
partitioned_df.createOrReplaceTempView("housing_partitioned")

print("Partitions on disk:")
for root, dirs, files in os.walk(OUT):
    level = root.replace(OUT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if files:
        label = f"{files[0]} (+ {len(files)-1} more)" if len(files) > 1 else files[0]
        print(f"{indent}  {label}")


In [ ]:
# Partition pruning — only the 'furnished' directory is read
spark.sql("""
    SELECT COUNT(*), ROUND(AVG(price),0) AS avg_price
    FROM   housing_partitioned
    WHERE  furnishingstatus = 'furnished'
""").show()


In [ ]:
# Inspect the physical plan — look for 'PartitionFilters' in the output
spark.sql("""
    SELECT COUNT(*) FROM housing_partitioned
    WHERE  furnishingstatus = 'furnished'
""").explain("simple")


---
## 8. Window Functions

Window functions compute a value for each row **relative to a group of rows** (the *window*), without collapsing rows the way `GROUP BY` does.

```sql
function()  OVER (
    PARTITION BY col1           -- defines the group (like GROUP BY, but rows are kept)
    ORDER BY     col2           -- ordering within the group
    ROWS / RANGE BETWEEN ...   -- optional frame specification
)
```

| Function | Purpose |
|----------|---------|
| `ROW_NUMBER()` | Unique sequential rank (no ties) |
| `RANK()` | Rank with gaps after ties |
| `DENSE_RANK()` | Rank without gaps |
| `LAG(col, n)` / `LEAD(col, n)` | Access a value n rows before / after the current row |
| `SUM / AVG / MIN / MAX` | Running or rolling aggregates |
| `NTILE(n)` | Divide rows into n equal buckets |
| `PERCENT_RANK()` | Relative rank as a fraction between 0 and 1 |


In [ ]:
# ROW_NUMBER — rank houses by price within each furnishing category
spark.sql("""
    SELECT
        price,
        area,
        bedrooms,
        furnishingstatus,
        ROW_NUMBER() OVER (
            PARTITION BY furnishingstatus
            ORDER BY price DESC
        ) AS rank_in_category
    FROM housing
    QUALIFY rank_in_category <= 3
    ORDER BY furnishingstatus, rank_in_category
""").show()


`QUALIFY` filters on window function results inline, the same way `HAVING` filters on aggregate results. It avoids the need to wrap the query in a subquery just to filter on the rank.


In [ ]:
# Running total and 5-row moving average of price ordered by area
spark.sql("""
    SELECT
        price,
        area,
        bedrooms,
        SUM(price)  OVER (ORDER BY area
                          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                         )                        AS running_total_price,
        ROUND(
          AVG(price) OVER (ORDER BY area
                           ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
          0)                                      AS moving_avg_5
    FROM housing
    ORDER BY area
    LIMIT 15
""").show()


In [ ]:
# NTILE — split houses into price quartiles
spark.sql("""
    SELECT
        price,
        area,
        furnishingstatus,
        NTILE(4) OVER (ORDER BY price) AS price_quartile
    FROM housing
    ORDER BY price
    LIMIT 20
""").show()


In [ ]:
# LAG / LEAD — compare each house price with its neighbours in price order
spark.sql("""
    SELECT
        price,
        area,
        LAG(price, 1)  OVER (ORDER BY price) AS prev_price,
        LEAD(price, 1) OVER (ORDER BY price) AS next_price,
        price - LAG(price,1) OVER (ORDER BY price) AS gap_from_prev
    FROM housing
    ORDER BY price
    LIMIT 15
""").show()


---
## 9. User-Defined Functions (UDFs)

When built-in Spark functions are not enough, you can register a Python function as a **UDF** and call it from SQL.

> ⚠️ **Performance warning** — Python UDFs cross the JVM-to-Python boundary row by row, serialising and deserialising every value.
> For CPU-bound transformations, prefer **Pandas UDFs** (`@pandas_udf`), which operate on Apache Arrow batches and are typically 10–100x faster.

### 9.1 Scalar UDF


In [ ]:
from pyspark.sql.types import StringType, DoubleType

def price_label(price):
    """Return a human-readable price tier label."""
    if price is None:
        return "Unknown"
    if price < 3_500_000:
        return "Budget"
    if price < 6_000_000:
        return "Mid-range"
    if price < 9_000_000:
        return "Premium"
    return "Luxury"

spark.udf.register("price_label", price_label, StringType())

spark.sql("""
    SELECT
        price,
        price_label(price) AS tier,
        area,
        bedrooms
    FROM housing
    LIMIT 12
""").show()


### 9.2 Pandas UDF (Vectorised)

A Pandas UDF receives an entire column (or multiple columns) as a `pandas.Series` and returns a `Series`. Spark transfers data between the JVM and Python using Apache Arrow, which is a columnar in-memory format — far more efficient than row-by-row serialisation.


In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(DoubleType())
def price_per_sqft(price: pd.Series, area: pd.Series) -> pd.Series:
    """Price per square foot — vectorised over entire columns at once."""
    return (price / area).round(2)

spark.udf.register("price_per_sqft", price_per_sqft)

spark.sql("""
    SELECT
        price,
        area,
        price_per_sqft(price, area)  AS ppsf,
        furnishingstatus
    FROM housing
    ORDER BY ppsf DESC
    LIMIT 10
""").show()


### 9.3 Combining UDFs with Window Functions


In [ ]:
spark.sql("""
    SELECT
        price,
        area,
        furnishingstatus,
        price_label(price)               AS tier,
        price_per_sqft(price, area)      AS ppsf,
        ROUND(
          AVG(price_per_sqft(price, area))
              OVER (PARTITION BY furnishingstatus), 2
        )                                AS avg_ppsf_by_furnishing
    FROM housing
    ORDER BY furnishingstatus, ppsf DESC
    LIMIT 15
""").show()


---
# Exercises — Penn World Table v11.0

The **Penn World Table** (PWT) is one of the most authoritative macroeconomic datasets, providing purchasing-power-adjusted GDP, population, employment, capital stock, and total factor productivity for 183 countries from 1950 to 2023.

The data is provided as a Parquet file (`penn_world_table.parquet`). Parquet is a columnar binary format and the default storage format in Spark pipelines — it preserves type information, supports predicate pushdown, and is significantly more efficient to read than CSV or Excel for large datasets.

**Key columns used in the exercises**

| Column | Description |
|--------|-------------|
| `countrycode` | ISO 3-letter code |
| `country` | Country name |
| `year` | Year (1950–2023) |
| `rgdpe` | Real GDP (expenditure-side), millions of 2017 USD PPP |
| `rgdpo` | Real GDP (output-side), millions of 2017 USD PPP |
| `pop` | Population, millions |
| `emp` | Number of persons engaged, millions |
| `hc` | Human capital index (based on education and returns to schooling) |
| `ctfp` | Total factor productivity relative to the US frontier |
| `labsh` | Labour share of income |


In [ ]:
pwt_df = spark.read.parquet("penn_world_table.parquet")

pwt_df.createOrReplaceTempView("pwt")

print(f"Rows: {pwt_df.count():,}  |  Columns: {len(pwt_df.columns)}")
pwt_df.select("countrycode", "country", "year", "rgdpe", "pop", "hc", "labsh").show(5)


In [ ]:
# Cache PWT — we will query it repeatedly throughout the exercises
spark.sql("CACHE TABLE pwt")

---
## Exercise 1 — Basic Aggregation: GDP per Capita Rankings

**Goal**: Find the top 10 countries by GDP per capita for the year 2019. Only include countries with at least 1 million inhabitants.

**Concepts**: `WHERE`, computed columns, `ORDER BY`, `LIMIT`

---
### Your query


In [ ]:
# Hints:
#   - GDP per capita = rgdpe / pop  (both in millions, so the units cancel to thousands of USD)
#   - Filter: year = 2019 AND pop >= 1 AND rgdpe IS NOT NULL
#   - Order by GDP per capita descending

spark.sql("""
    -- YOUR QUERY HERE
""").show(10, truncate=False)


### Solution

In [ ]:
spark.sql("""
    SELECT
        country,
        countrycode,
        ROUND(rgdpe / pop, 2)  AS gdp_per_capita_k_usd,
        ROUND(pop, 1)          AS pop_millions
    FROM   pwt
    WHERE  year  = 2019
      AND  pop   >= 1
      AND  rgdpe IS NOT NULL
    ORDER  BY gdp_per_capita_k_usd DESC
    LIMIT  10
""").show(10, truncate=False)


---
## Exercise 2 — Window Functions: Ranking Countries by GDP Growth

**Goal**: For each country, compute the year-over-year GDP growth rate using `LAG`. Then find the top 10 countries by average annual growth rate since 1990, among those with at least 20 years of data.

**Concepts**: `LAG()`, `NULLIF`, `GROUP BY`, `HAVING`, `ORDER BY`

---
### Your query


In [ ]:
# Hints:
#   - Compute yoy_growth in a CTE using LAG(rgdpe) OVER (PARTITION BY countrycode ORDER BY year)
#   - Growth rate = (rgdpe - prev_rgdpe) / NULLIF(prev_rgdpe, 0)
#   - In the outer query, GROUP BY country and filter with HAVING COUNT(*) >= 20
#   - Order by average growth descending

spark.sql("""
    -- YOUR QUERY HERE
""").show(10, truncate=False)


### Solution

In [ ]:
spark.sql("""
    WITH growth AS (
        SELECT
            country,
            year,
            (rgdpe - LAG(rgdpe) OVER (PARTITION BY countrycode ORDER BY year))
              / NULLIF(LAG(rgdpe) OVER (PARTITION BY countrycode ORDER BY year), 0)
                                AS yoy_growth
        FROM pwt
        WHERE rgdpe IS NOT NULL
          AND year  >= 1990
    )
    SELECT
        country,
        ROUND(AVG(yoy_growth) * 100, 2)  AS avg_growth_pct,
        COUNT(*)                         AS years_of_data
    FROM   growth
    WHERE  yoy_growth IS NOT NULL
    GROUP  BY country
    HAVING COUNT(*) >= 20
    ORDER  BY avg_growth_pct DESC
    LIMIT  10
""").show(10, truncate=False)


---
## Exercise 3 — UDFs: Classifying Countries by GDP per Capita

**Goal**: Register a scalar UDF called `wealth_tier` that maps a country's GDP per capita (in thousands of 2017 USD) to one of four labels: `"Low"`, `"Middle"`, `"High"`, or `"Very High"`. Then use it in a SQL query to count how many countries fall into each tier in 2019, and show the average GDP per capita per tier.

**Thresholds** (thousands of 2017 USD per capita): `< 5` Low | `5–20` Middle | `20–60` High | `>= 60` Very High

**Concepts**: `spark.udf.register`, calling a UDF in SQL, `GROUP BY`

---
### Your query


In [ ]:
from pyspark.sql.types import StringType

# Define a Python function and register it as a Spark SQL UDF
def wealth_tier(gdp_per_capita):
    # YOUR LOGIC HERE
    pass

spark.udf.register("wealth_tier", wealth_tier, StringType())

spark.sql("""
    -- YOUR QUERY HERE
    -- Hint: GDP per capita = rgdpe / pop, filter year = 2019 and pop >= 1
""").show(truncate=False)


### Solution

In [ ]:
from pyspark.sql.types import StringType

def wealth_tier(gdp_per_capita):
    if gdp_per_capita is None:
        return "Unknown"
    if gdp_per_capita < 5_000:
        return "Low"
    if gdp_per_capita < 20_000:
        return "Middle"
    if gdp_per_capita < 60_000:
        return "High"
    return "Very High"

spark.udf.register("wealth_tier", wealth_tier, StringType())

spark.sql("""
    SELECT
        wealth_tier(ROUND(rgdpe / pop, 2))  AS tier,
        COUNT(*)                            AS num_countries,
        ROUND(AVG(rgdpe / pop), 2)          AS avg_gdp_per_capita_k_usd
    FROM   pwt
    WHERE  year  = 2019
      AND  pop   >= 1
      AND  rgdpe IS NOT NULL
    GROUP  BY tier
    ORDER  BY avg_gdp_per_capita_k_usd DESC
""").show(truncate=False)


---
## Exercise 4 — Caching and Window Functions: GDP per Capita Over Time

**Goal**: Compute each country's GDP per capita for every year, then use a window function to add a column showing the previous year's value with `LAG`. Cache the resulting view, then query it to find the 10 countries with the highest average year-over-year increase in GDP per capita since 2000.

**Concepts**: `CREATE OR REPLACE TEMP VIEW`, `CACHE TABLE`, `LAG`, `GROUP BY`, `ORDER BY`

---
### Your query


In [ ]:
# Step 1 -- create and cache a view with GDP per capita and the lagged value
# Hint: CREATE OR REPLACE TEMP VIEW gdp_pc AS SELECT ...
#       include LAG(rgdpe / pop) OVER (PARTITION BY countrycode ORDER BY year) AS prev_gdp_pc

# Step 2 -- cache the view
# spark.sql("CACHE TABLE gdp_pc")

# Step 3 -- query the cached view
# Find the 10 countries with the highest average annual increase since 2000
# Hint: increase = (rgdpe / pop) - prev_gdp_pc, then AVG, GROUP BY country

spark.sql("""
    -- YOUR QUERY HERE
""").show(10, truncate=False)


### Solution

In [ ]:
# Step 1 -- create a view with GDP per capita and its lagged value
spark.sql("""
    CREATE OR REPLACE TEMP VIEW gdp_pc AS
    SELECT
        country,
        countrycode,
        year,
        ROUND(rgdpe / pop, 2)  AS gdp_per_capita,
        LAG(ROUND(rgdpe / pop, 2)) OVER (
            PARTITION BY countrycode ORDER BY year
        )                      AS prev_gdp_per_capita
    FROM pwt
    WHERE rgdpe IS NOT NULL
      AND pop   IS NOT NULL
""")

# Step 2 -- cache the view
spark.sql("CACHE TABLE gdp_pc")

In [ ]:
# Step 3 -- query the cached view
spark.sql("""
    SELECT
        country,
        ROUND(AVG(gdp_per_capita - prev_gdp_per_capita), 2)  AS avg_annual_increase,
        COUNT(*)                                             AS years_of_data
    FROM   gdp_pc
    WHERE  year >= 2000
      AND  prev_gdp_per_capita IS NOT NULL
    GROUP  BY country
    ORDER  BY avg_annual_increase DESC
    LIMIT  10
""").show(10, truncate=False)


---
## Bonus Exercise 5 — Full Pipeline: Ranking Countries by Human Capital

**Goal**: Register a Pandas UDF called `hc_score` that takes the human capital index (`hc`) and the labour share of income (`labsh`) and returns a composite score: `0.6 * hc + 0.4 * labsh`. Then write a SQL query that computes this score for each country in 2019 and uses `PERCENT_RANK()` to show where each country stands globally. Show the top 15 countries.

**Concepts**: `@pandas_udf`, `PERCENT_RANK()` as a window function, `ORDER BY`

---
### Your solution


In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType

@pandas_udf(DoubleType())
def hc_score(hc: pd.Series, labsh: pd.Series) -> pd.Series:
    return (0.6 * hc + 0.4 * labsh).round(4)

spark.udf.register("hc_score", hc_score)

spark.sql("""
    SELECT
        country,
        ROUND(hc_score(hc, labsh), 4)                                AS score,
        ROUND(PERCENT_RANK() OVER (ORDER BY hc_score(hc, labsh)), 3) AS pct_rank
    FROM   pwt
    WHERE  year  = 2019
      AND  hc    IS NOT NULL
      AND  labsh IS NOT NULL
    ORDER  BY score DESC
    LIMIT  15
""").show(15, truncate=False)


In [ ]:
# -- Wrap-up
spark.sql("UNCACHE TABLE pwt")
spark.sql("UNCACHE TABLE gdp_pc")
spark.stop()